In [1]:
# ============================================================================
# STEP 1: ENVIRONMENT SETUP & API KEY CONFIGURATION
# ============================================================================
# This cell handles .env file creation and API key setup
# RUN THIS FIRST!

import os
from pathlib import Path
import getpass
import datetime
import os
import sys
import getpass
from pathlib import Path


# ============================================================================
# SECURE KEY MANAGER (add this function)
# ============================================================================
_SECURE_KEYS = {}

def secure_get_key(service_name, key_name, prompt_message=None):
    """Get API key securely - stored only in memory"""
    global _SECURE_KEYS
    
    # Return from memory if already entered
    if key_name in _SECURE_KEYS:
        return _SECURE_KEYS[key_name]
    
    # Prompt user for key
    if prompt_message is None:
        prompt_message = f"Enter your {service_name} API key: "
    
    print(f"\n🔐 {service_name} API Key Required")
    print("-" * 50)
    key = getpass.getpass(prompt_message)
    
    if not key:
        raise ValueError(f"No API key provided for {service_name}")
    
    # Store in memory only
    _SECURE_KEYS[key_name] = key
    
    print(f"✓ {service_name} API key loaded (memory only)")
    return key


LOG_DIR = "log_folder"

# Create the folder if it does not exist
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

# Combine the folder path and the file name
LOG_FILE = os.path.join(LOG_DIR, f"propose_log_{datetime.datetime.now():%Y%m%d_%H%M%S}.txt")

def log_to_file(msg):
    with open(LOG_FILE, "a") as f:
        f.write(msg + "\n")

In [2]:
# ============================================================================
# STEP 2: IMPORTS & OPENAI API SETUP
# ============================================================================

import re
import json
import sympy
import numpy as np
from datetime import datetime
from typing import List, Dict, Tuple, Any
import itertools
import time
import sys
from io import StringIO
import ast
import math
from itertools import combinations
import threading
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
# Configure openai API
from openai import OpenAI
DEBUG_FILE = "debug_solve.log"

def debug_log(msg):
    with open(DEBUG_FILE, "a") as f:
        f.write(msg + "\n")
print("\n📌 DeepSeek Model Selection:")
print("  1. deepseek-chat (standard, lower cost, good for reasoning with explicit prompts)")
print("  2. deepseek-reasoner (verbose, shows internal reasoning, higher token cost)")
model_choice = input("Select model (1 (chat) or 2 (reasoner), default: 1): ").strip() or "1"

if model_choice == "2":
    DEEPSEEK_MODEL = "deepseek-reasoner"
    print("✓ Using deepseek-reasoner (verbose reasoning)")
else:
    DEEPSEEK_MODEL = "deepseek-chat"
    print("✓ Using deepseek-chat (standard)")
# Get DeepSeek API key securely
DEEPSEEK_API_KEY = secure_get_key("DeepSeek", "DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = input("\nDeepSeek base URL (press Enter for default): ").strip()
if not DEEPSEEK_BASE_URL:
    DEEPSEEK_BASE_URL = "https://api.deepseek.com"

# Initialize DeepSeek client
deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

print(f"✓ DeepSeek client configured")
print(f"  Base URL: {DEEPSEEK_BASE_URL}")

# Rate limiting configuration for free tier
# Free tier limits: 20 requests/minute, 14k requests/day
API_DELAY = 0.01  # 0.01 seconds between requests = ~60 requests/minute (safe margin)
last_api_call_time = 0



_last_api_call_time = 0
# _api_lock = threading.Lock()

# def rate_limited_api_call():
#     global _last_api_call_time
#     with _api_lock:
#         current_time = time.time()
#         time_since_last = current_time - _last_api_call_time
#         if time_since_last < API_DELAY:
#             sleep_time = API_DELAY - time_since_last
#             time.sleep(sleep_time)
#         _last_api_call_time = time.time()

def deepseek_generate(prompt: str, n: int = 1, temperature: float = 0.7, max_tokens: int = 1500, stats: dict = None) -> List[str]:
    """
    Generate text using DeepSeek API. If `stats` dict is provided, it will accumulate token usage.
    stats dict should have keys 'total_input_tokens' and 'total_output_tokens'.
    """
    outputs = []
    for _ in range(n):
        response_text = ""
        for attempt in range(3):
            try:
                time.sleep(API_DELAY)
                response = deepseek_client.chat.completions.create(
                    model=DEEPSEEK_MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                text = response.choices[0].message.content
                if text and text.strip():
                    response_text = text.strip()
                    # Track token usage
                    if stats is not None:
                        usage = response.usage
                        stats['total_input_tokens'] = stats.get('total_input_tokens', 0) + usage.prompt_tokens
                        stats['total_output_tokens'] = stats.get('total_output_tokens', 0) + usage.completion_tokens
                    break
                else:
                    time.sleep(1)
            except Exception as e:
                log_to_file(f"DeepSeek attempt {attempt+1} error: {e}")
                if "429" in str(e) or "rate" in str(e).lower():
                    time.sleep(60)
                else:
                    time.sleep(1)
        outputs.append(response_text)
    return outputs

print(f"✓ Rate limiting configured: {API_DELAY}s delay (~{int(60/API_DELAY)} req/min)")
print("✓ DeepSeek API setup complete")


📌 DeepSeek Model Selection:
  1. deepseek-chat (standard, lower cost, good for reasoning with explicit prompts)
  2. deepseek-reasoner (verbose, shows internal reasoning, higher token cost)
✓ Using deepseek-chat (standard)

🔐 DeepSeek API Key Required
--------------------------------------------------
✓ DeepSeek API key loaded (memory only)
✓ DeepSeek client configured
  Base URL: https://api.deepseek.com
✓ Rate limiting configured: 0.01s delay (~6000 req/min)
✓ DeepSeek API setup complete


In [3]:
# ============================================================================
# STEP 3: SAFE SANDBOX FOR CODE EXECUTION
# ============================================================================

class SafeAgentSandbox:
    """A SECURE sandbox that only allows safe math operations"""
    def __init__(self):
        # ONLY allow these safe built-ins
        self.safe_builtins = {
            'print': print,
            'range': range,
            'len': len,
            'sum': sum,
            'min': min,
            'max': max,
            'abs': abs,
            'round': round,
            'int': int,
            'float': float,
            'str': str,
            'list': list,
        }
        self.globals = {
            "__builtins__": self.safe_builtins,
            "math": __import__("math"),
        }
        
    def run(self, code):
        """Execute code in restricted environment with validation"""
        # Blacklist dangerous operations
        dangerous_keywords = ['import', 'open', 'exec', 'eval', '__', 'os', 'sys', 'subprocess', 'file']
        if any(keyword in code.lower() for keyword in dangerous_keywords):
            return "Error: Code contains forbidden operations", None
        
        # Check for number reuse (e.g., numbers[1] + numbers[1])
        index_pattern = r'numbers\[(\d+)\]'
        indices_used = re.findall(index_pattern, code)
        if len(indices_used) != len(set(indices_used)):
            return "Error: Cannot use the same number twice in one operation", None
        
        # Prevent using arbitrary numbers not in the 'numbers' array
        lines = code.split('\n')
        for line in lines:
            if '#' in line:
                line = line.split('#')[0]
            line = line.strip()
            
            if not line or line.startswith('numbers = ') or line.startswith('remaining = ') or line.startswith('new_numbers = ') or line.startswith('print('):
                continue
                
            if line.startswith('res = ') and any(op in line for op in [' + ', ' - ', ' * ', ' / ', ' % ', ' // ']):
                rhs = line.split('res = ', 1)[1]
                numeric_literals = re.findall(r'(?<!\[)\b(\d+(?:\.\d+)?)\b(?!\])', rhs)
                
                for num_str in numeric_literals:
                    if f'[{num_str}]' in rhs:
                        continue
                    return f"Error: Cannot use arbitrary number {num_str} - must use only numbers from the array via numbers[index]", None
        
        try:
            output_buffer = StringIO()
            sys.stdout = output_buffer
            
            exec(code, self.globals)
            
            sys.stdout = sys.__stdout__
            output = output_buffer.getvalue().strip() or "Code executed successfully."
            
            new_state = self.globals.get('new_numbers', None)
            
            original_count = len(self.globals.get('numbers', []))
            new_count = len(new_state) if new_state else 0
            expected_count = original_count - 1
            
            if new_state and new_count != expected_count:
                return f"Error: Invalid operation - expected {expected_count} numbers, got {new_count}", None
            
            return output, new_state
            
        except Exception as e:
            sys.stdout = sys.__stdout__
            return f"Error: {str(e)}", None

# Initialize global sandbox
sandbox = SafeAgentSandbox()

# Test the sandbox
print("✓ Safe Sandbox initialized")
test_code = """
numbers = [4, 5, 6, 10]
res = numbers[0] + numbers[1]
remaining = numbers[2:]
new_numbers = [res] + remaining
print(new_numbers)
"""
# test_output, test_state = sandbox.run(test_code)
# print(f"Test result: {test_output}")

✓ Safe Sandbox initialized


In [4]:
# ============================================================================
# STEP 4: PROMPTS FOR CODEACT PATTERN
# ============================================================================

PROPOSE_PROMPT_CODEACT = PROPOSE_PROMPT_CODEACT = """
You are solving Game of 24 using executable Python code. Generate 8–10 possible next steps.

**CRITICAL CODE FORMAT (MUST FOLLOW EXACTLY)**:

You MUST write code that uses the list `numbers` and its indices (e.g., numbers[0], numbers[1]).
Do NOT use literal numbers like 4 or 6 in the operation – always use numbers[index].

Example for input [4, 6, 7, 10] (correct code):
numbers = [4, 6, 7, 10]
res = numbers[3] - numbers[1]   # 10 - 6 = 4
remaining = [numbers[0], numbers[2]]
new_numbers = [res] + remaining
print(new_numbers)

Note: The variable names `res`, `remaining`, `new_numbers` are mandatory.

Now generate steps for the current state.

Original puzzle: {original_input}
Steps taken so far:
{history}

Current numbers: {input}

For each step, provide:
Thought: [description]
Math: [calculation, e.g., "10 - 6 = 4"]
Remaining: [list of numbers after operation]
Code: (python code using the exact format above)

IMPORTANT:
- Try all pairs of indices.
- For subtraction and division, include both orders.
- Avoid dead ends: [24, ...] before final step; huge numbers (>50) with tiny numbers (<2).

Possible next steps:
"""

VALUE_PROMPT_CODEACT = """Numbers: {input}
Target: 24

You are a clever math puzzler. Make at most 5 attempts. After each attempt, note if the result is too high, too low, or close. Show your reasoning in short plain‑text bullet points. Do not use LaTeX.

Examples:

[3,6,10]:
- Attempt 1: 10 + 6 = 16, then 16 + 3 = 19 (too low).
- Attempt 2: 10 * 3 = 30, then 30 - 6 = 24 (exact!).
ANSWER: sure

[2,2,7]:
- Attempt 1: 7 * 2 = 14, then 14 + 2 = 16 (too low).
- Attempt 2: 7 * 2 = 14, then 14 * 2 = 28 (too high, within 20-28).
- Attempt 3: 7 + 2 = 9, then 9 * 2 = 18 (still low). Best is 28 within range.
ANSWER: likely

[10,10,10]:
- Attempt 1: 10 + 10 = 20, then 20 + 10 = 30 (too high).
- Attempt 2: 10 + 10 = 20, then 20 - 10 = 10 (too low).
- Attempt 3: 10 * 10 = 100, then 100 / 10 = 10 (still low).
- Attempt 4: 10 * 10 = 100, then 100 - 10 = 90 (far high).
All results are 10, 30, or 90. None between 20-28.
ANSWER: impossible

[10, 14]:
- Attempt 1: 10 + 14 = 24 (exact!)
ANSWER: sure

Now solve for {input} with target 24. Use the same style.

IMPORTANT DECISION RULE (apply after your reasoning):
- If you found a result **within 0.001 of 24** (i.e., 23.999 to 24.001) → ANSWER: sure
- Else if you found any result between 20 and 28 (inclusive) → ANSWER: likely
- Else → ANSWER: impossible

After reasoning, write a blank line then "ANSWER: sure/likely/impossible"
"""

# ============================================================================
# CUSTOM TWO-NUMBER PROMPT (NEW)
# ============================================================================
# For terminal 2-number states: pick ONE operation that works best

TWO_NUMBER_PROPOSE_PROMPT = """FINAL STEP: You have exactly 2 numbers. Pick ONE operation that gets closest to 24.

Original puzzle: {original_input}
Path so far:
{history}

Current numbers: {input}
Let a = {a}, b = {b}

Evaluate all 6 possible operations:
1. a + b = {a} + {b} = {sum_result}
2. a - b = {a} - {b} = {diff_ab}
3. b - a = {b} - {a} = {diff_ba}
4. a * b = {a} * {b} = {prod_result}
5. a / b = {a} / {b} = {div_ab}
6. b / a = {b} / {a} = {div_ba}

PICK THE OPERATION that gets CLOSEST to 24 (even if not exact).
ALWAYS return ONE operation - no matter which is best.

Output format:
Thought: [Which operation gets closest to 24? Why?]
Selected operation: [ONLY ONE: a+b OR a-b OR b-a OR a*b OR a/b OR b/a]
Result: [the number this produces]
"""

print("✓ CodeAct prompts loaded successfully (including TWO_NUMBER_PROPOSE_PROMPT)")

✓ CodeAct prompts loaded successfully (including TWO_NUMBER_PROPOSE_PROMPT)


In [5]:
# ============================================================================
# STEP 5: TREE NODE CLASS & HELPER FUNCTIONS
# ============================================================================

class TreeNode:
    """Represents a node in the Game of 24 search tree"""
    node_counter = 0
    
    def __init__(self, state: List[float], parent=None, action: str = "", value: float = 0.0, depth: int = 0):
        self.id = TreeNode.node_counter
        TreeNode.node_counter += 1
        
        self.state = state
        self.parent = parent
        self.action = action
        self.value = value
        self.depth = depth
        self.children = {}
        self.is_solution = False
        self.is_pruned = False
        self.evaluation = None  # Store evaluation record for distillation
        
        # CodeAct specific
        self.thought = ""
        self.code = ""
        self.observation = ""
        self.path_history = ""
    
    def to_dict(self):
        """Convert node to dictionary for JSON serialization"""
        return {
            'id': self.id,
            'parent_id': self.parent.id if self.parent else None,
            'state': self.state,
            'action': self.action,
            'value': self.value,
            'depth': self.depth,
            'is_solution': self.is_solution,
            'is_pruned': self.is_pruned,
            'codeact': {
                'thought': self.thought,
                'code': self.code,
                'observation': self.observation
            },
            'path_history': self.path_history,
            'evaluation': self.evaluation,
            'num_children': len(self.children),
            'children': {child_id: child.to_dict() for child_id, child in self.children.items()}
        }

def check_solution(state: List[float]) -> Tuple[bool, str]:
    """Check if any single element equals 24"""
    if len(state) == 1:
        if abs(state[0] - 24.0) < 1e-6:
            return True, f"{state[0]} = 24 ✓"
    return False, ""

def state_signature(nums: List[float]) -> tuple:
    """
    Classify state for Delayed Fraction Preservation (DFP).
    
    Returns:
        (non_int_count, small_int_count) tuple
    """
    non_int = [x for x in nums if abs(x - round(x)) > 1e-6]
    small_int = [x for x in nums if abs(x - round(x)) < 1e-6 and x <= 6]
    return len(non_int), len(small_int)

print("✓ TreeNode and helper functions defined")

# ============================================================================
# STEP 5.5: DEAD-END MEMORY (TIM Idea #7 Integration)
# ============================================================================

class DeadEndMemory:
    """
    Improved dead-end memory combining three strategies:

    Direction 1 (3-number states): Store operation-sequence patterns + LLM reasoning text.
      Matching asks: did this candidate arrive via the same op-type sequence?

    Direction 2 (3-number states): Feed stored LLM reasoning text back into proposal
      prompt so the LLM avoids re-proposing structurally identical failures.

    Direction 3 (2-number states): Deterministic lookup cache — check all 6 ops in
      microseconds. No fingerprint matching needed; result is exact and cross-problem valid.

    Public interface is unchanged:
      record_dead_end(node, reason)
      is_potential_dead_end(new_state) -> (bool, matched_pattern|None)
      get_pattern_summary() -> str
      save(path) / load(path)
      current_problem_id  (set externally)
    """

    TARGET = 24.0

    def __init__(self, similarity_threshold=0.5, test_mode=False):
        # --- Direction 3: exact 2-number cache ---
        self.two_number_cache = {}        # (min,max) -> bool (can reach 24)

        # --- Direction 1+2: 3-number semantic patterns ---
        self.patterns = []                # list of pattern dicts

        self.similarity_threshold = similarity_threshold
        self.test_mode = test_mode
        self.problems_seen = 0
        self.current_problem_id = "unknown"

        # Stats (kept for compatibility)
        self.total_skipped = 0
        self.total_checked = 0

    # ------------------------------------------------------------------ #
    #  Direction 3 helpers                                                 #
    # ------------------------------------------------------------------ #

    def _two_number_can_reach_24(self, a, b):
        """Check all 6 operations exactly. Returns (can_reach, operation_str|None)."""
        ops = [
            (a + b, f"{a} + {b}"),
            (a - b, f"{a} - {b}"),
            (b - a, f"{b} - {a}"),
            (a * b, f"{a} * {b}"),
        ]
        if abs(b) > 1e-9:
            ops.append((a / b, f"{a} / {b}"))
        if abs(a) > 1e-9:
            ops.append((b / a, f"{b} / {a}"))
        for result, op_str in ops:
            if abs(result - self.TARGET) < 0.001:
                return True, op_str
        return False, None

    def _two_number_lookup(self, state):
        """Return (is_dead_end, reason_str) for a 2-number state."""
        a, b = float(state[0]), float(state[1])
        key = (min(a, b), max(a, b))
        if key not in self.two_number_cache:
            can_reach, op = self._two_number_can_reach_24(a, b)
            self.two_number_cache[key] = (can_reach, op)
        can_reach, op = self.two_number_cache[key]
        if can_reach:
            return False, None          # not a dead end
        return True, {"reason": f"no_op_reaches_24: tried all 6 ops on {a},{b}",
                      "num_count": 2,
                      "thought": f"Deterministic: no operation between {a} and {b} reaches 24",
                      "hit_count": 99,
                      "problems_seen_in": ["__deterministic__"]}

    # ------------------------------------------------------------------ #
    #  Direction 1 helpers                                                 #
    # ------------------------------------------------------------------ #

    def _op_type(self, action_str):
        """Classify an action string into a coarse op type."""
        if not action_str:
            return "unknown"
        s = str(action_str).strip()
        if '*' in s:  return 'mul'
        if '/' in s:  return 'div'
        if '+' in s:  return 'add'
        if '-' in s:  return 'sub'
        return 'unknown'

    def _get_op_sequence(self, node):
        """Walk from root to node collecting op types."""
        seq = []
        cur = node
        while cur is not None and cur.parent is not None:
            seq.append(self._op_type(cur.action))
            cur = cur.parent
        return list(reversed(seq))

    def _extract_3num_pattern(self, node, reason):
        """Build a pattern dict for a 3-number dead-end node (Dir 1+2)."""
        state = node.state
        abs_vals = sorted([abs(x) for x in state], reverse=True)
        op_seq   = self._get_op_sequence(node)

        # Best reasoning text from eval_record (Dir 2)
        thought = ""
        if node.evaluation:
            reasoning = node.evaluation.get('reasoning', [])
            # Prefer the last non-trivial line
            for r in reversed(reasoning):
                if len(str(r)) > 20:
                    thought = str(r)
                    break

        return {
            'num_count':      len(state),
            'sorted_vals':    abs_vals,           # [max, mid, min]
            'op_sequence':    op_seq,
            'has_huge':       any(abs(x) > 50 for x in state),
            'has_tiny':       any(0 < abs(x) < 2 for x in state),
            'sum':            sum(abs(x) for x in state),
            'thought':        thought,             # Dir 2: LLM reasoning text
            'reason':         reason,
            'found_at_depth': node.depth,
            'value_score':    node.value,
            'hit_count':      1,
            'problems_seen_in': [self.current_problem_id],
        }

    def _3num_patterns_similar(self, new_vals, new_ops, stored):
        """
        Match a candidate (new_vals, new_ops) against a stored 3-number pattern.
        new_vals: sorted abs values of candidate state  [max, mid, min]
        new_ops:  op sequence that produced this candidate
        """
        sv = stored['sorted_vals']
        if len(new_vals) != len(sv):
            return False

        # Values: each element within ±25%
        for nv, pv in zip(new_vals, sv):
            ratio = nv / (pv + 1e-9)
            if ratio < 0.75 or ratio > 1.25:
                return False

        # Sum within ±25%
        new_sum = sum(new_vals)
        p_sum   = stored['sum']
        if abs(new_sum - p_sum) / (p_sum + 1e-9) > 0.25:
            return False

        # has_huge / has_tiny must match
        new_huge = any(v > 50 for v in new_vals)
        new_tiny = any(0 < v < 2 for v in new_vals)
        if new_huge != stored['has_huge'] or new_tiny != stored['has_tiny']:
            return False

        # Op sequence: must share at least the first step if both have ops
        stored_ops = stored.get('op_sequence', [])
        if new_ops and stored_ops:
            if new_ops[0] != stored_ops[0]:
                return False

        return True

    # ------------------------------------------------------------------ #
    #  Public API                                                          #
    # ------------------------------------------------------------------ #

    def record_dead_end(self, node, reason=""):
        """Store a dead-end pattern. Only records value=0.001 (impossible) nodes."""
        if node.value > 0.001:
            return

        # Direction 3: 2-number states are handled by deterministic cache — no need to store
        if len(node.state) == 2:
            a, b = float(node.state[0]), float(node.state[1])
            key = (min(a, b), max(a, b))
            # Prime the cache if not already there
            if key not in self.two_number_cache:
                can_reach, op = self._two_number_can_reach_24(a, b)
                self.two_number_cache[key] = (can_reach, op)
            return   # don't add to patterns list

        # Direction 1+2: 3-number patterns
        new_pat = self._extract_3num_pattern(node, reason)

        # Deduplicate: increment existing similar pattern
        for existing in self.patterns:
            if self._3num_patterns_similar(
                new_pat['sorted_vals'], new_pat['op_sequence'], existing
            ):
                existing['hit_count'] += 1
                if self.current_problem_id not in existing.get('problems_seen_in', []):
                    existing['problems_seen_in'].append(self.current_problem_id)
                # Update thought if new one is richer
                if len(new_pat['thought']) > len(existing.get('thought', '')):
                    existing['thought'] = new_pat['thought']
                return

        self.patterns.append(new_pat)

    def is_potential_dead_end(self, new_state):
        """
        Returns (is_dead_end: bool, matched_pattern: dict | None).
        Interface unchanged from original.
        """
        self.total_checked += 1

        if not new_state:
            return False, None

        # Direction 3: exact check for 2-number states
        if len(new_state) == 2:
            is_dead, pattern = self._two_number_lookup(new_state)
            if is_dead:
                self.total_skipped += 1
                return True, pattern
            return False, None

        # Direction 1: 3-number pattern match
        abs_vals = sorted([abs(x) for x in new_state], reverse=True)
        # We don't know the op sequence of the candidate at check time
        # (it hasn't been created yet), so we only use value/structural matching
        for stored in self.patterns:
            if not self._3num_patterns_similar(abs_vals, [], stored):
                continue
            # Trust gate: require cross-problem confirmation in normal mode
            if self.test_mode:
                if stored.get('hit_count', 1) < 2:
                    continue
            else:
                if len(stored.get('problems_seen_in', [])) < 2:
                    continue
            self.total_skipped += 1
            return True, stored

        return False, None

    def get_pattern_summary(self, max_patterns=3):
        """
        Direction 2: inject stored LLM reasoning text into proposal prompt.
        Returns a string ready to append to the proposal prompt.
        """
        # Collect 3-number patterns that have a useful thought
        useful = [
            p for p in self.patterns
            if p.get('thought') and p.get('found_at_depth', 0) >= 1
        ]
        if not useful:
            return ""

        # Sort by trust (hit_count desc) then worst value
        useful.sort(key=lambda p: (-p.get('hit_count', 1), p.get('value_score', 1.0)))
        chosen = useful[:max_patterns]

        lines = ["LEARNED DEAD-END PATTERNS (avoid repeating these failures):"]
        for i, p in enumerate(chosen, 1):
            vals = p.get('sorted_vals', [])
            ops  = p.get('op_sequence', [])
            thought = p.get('thought', '')
            op_str = " → ".join(ops) if ops else "unknown ops"
            lines.append(f"\nPattern {i}: state ~{vals} via [{op_str}]")
            lines.append(f"  LLM verdict: {thought}")
            lines.append(f"  → AVOID proposals that produce similar number combinations via {op_str}")
        return "\n".join(lines)

    def get_stats(self):
        return {
            'patterns_stored':    len(self.patterns),
            'two_num_cache_size': len(self.two_number_cache),
            'total_checked':      self.total_checked,
            'total_skipped':      self.total_skipped,
            'skip_rate':          f"{100*self.total_skipped/max(1,self.total_checked):.1f}%"
        }

    def save(self, path):
        if self.test_mode:
            return
        import json, os
        data = {
            'patterns':       self.patterns,
            'two_num_cache':  {f"{k[0]},{k[1]}": v for k, v in self.two_number_cache.items()},
            'problems_seen':  self.problems_seen + 1,
        }
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)

    @classmethod
    def load(cls, path, test_mode=False):
        import json, os
        mem = cls(test_mode=test_mode)
        if os.path.exists(path):
            with open(path) as f:
                data = json.load(f)
            mem.patterns      = data.get('patterns', [])
            mem.problems_seen = data.get('problems_seen', 0)
            for k_str, v in data.get('two_num_cache', {}).items():
                parts = k_str.split(',')
                key   = (float(parts[0]), float(parts[1]))
                mem.two_number_cache[key] = tuple(v) if isinstance(v, list) else v
        return mem


print("✓ DeadEndMemory (Dir 1+2+3) loaded")

# ============================================================================
# STEP 5.6: INCONSISTENCY DETECTION (TIM Idea #4)
# ============================================================================

class InconsistencyDetector:
    """
    Detects when the same game state receives inconsistent evaluation scores.
    
    TiM Idea #4: If same state gets very different scores on re-evaluation,
    the LLM evaluation is unreliable.
    
    Tracks all evaluations and flags states with high variance.
    """
    
    def __init__(self, inconsistency_threshold=0.3):
        """
        Args:
            inconsistency_threshold: If max_score - min_score > threshold,
                                     flag as inconsistent (0-1 scale)
        """
        self.evaluation_history = {}  # state_tuple → list of scores
        self.inconsistency_threshold = inconsistency_threshold
        self.inconsistent_states = []
        self.total_evaluations = 0
        self.re_evaluations = 0
    
    def record_evaluation(self, state, score, depth=0, reasoning=""):
        """Record an evaluation of a state"""
        self.total_evaluations += 1
        
        # Create a hashable key (sorted tuple)
        state_key = tuple(sorted([round(x, 6) for x in state]))
        
        if state_key not in self.evaluation_history:
            self.evaluation_history[state_key] = []
        
        self.evaluation_history[state_key].append({
            'score': score,
            'depth': depth,
            'reasoning': reasoning,
            'timestamp': datetime.now()
        })
        
        # Check for inconsistency if we've seen this state before
        if len(self.evaluation_history[state_key]) > 1:
            self.re_evaluations += 1
            return self._check_inconsistency(state_key)
        
        return None
    
    def _check_inconsistency(self, state_key):
        """Check if evaluations of this state are inconsistent"""
        evaluations = self.evaluation_history[state_key]
        scores = [e['score'] for e in evaluations]
        
        min_score = min(scores)
        max_score = max(scores)
        variance = max_score - min_score
        
        # Flag if variance exceeds threshold
        if variance > self.inconsistency_threshold:
            info = {
                'state': list(state_key),
                'is_inconsistent': True,
                'num_evaluations': len(evaluations),
                'scores': scores,
                'min_score': min_score,
                'max_score': max_score,
                'variance': variance,
                'depths': [e['depth'] for e in evaluations]
            }
            self.inconsistent_states.append(info)
            return info
        
        return {
            'state': list(state_key),
            'is_inconsistent': False,
            'num_evaluations': len(evaluations),
            'scores': scores,
            'variance': variance
        }
    
    def get_inconsistent_states(self):
        """Get list of all states with inconsistent evaluations"""
        return self.inconsistent_states
    
    def get_suspicious_states(self, min_variance=0.15):
        """Get states with moderate variance (suspicious but not flagged)"""
        suspicious = []
        for state_key, evals in self.evaluation_history.items():
            if len(evals) > 1:
                scores = [e['score'] for e in evals]
                variance = max(scores) - min(scores)
                
                if min_variance <= variance <= self.inconsistency_threshold:
                    suspicious.append({
                        'state': list(state_key),
                        'num_evaluations': len(evals),
                        'scores': scores,
                        'variance': variance
                    })
        
        return suspicious
    
    def get_stats(self):
        """Return statistics about evaluations"""
        return {
            'total_evaluations': self.total_evaluations,
            're_evaluations': self.re_evaluations,
            'unique_states': len(self.evaluation_history),
            'inconsistent_states': len(self.inconsistent_states),
            're_evaluation_rate': f"{100*self.re_evaluations/max(1, self.total_evaluations):.1f}%" if self.total_evaluations > 0 else "0%",
            'inconsistency_rate': f"{100*len(self.inconsistent_states)/max(1, self.re_evaluations):.1f}%" if self.re_evaluations > 0 else "0%"
        }

print("✓ InconsistencyDetector class (TIM Idea #4) loaded")



✓ TreeNode and helper functions defined
✓ DeadEndMemory (Dir 1+2+3) loaded
✓ InconsistencyDetector class (TIM Idea #4) loaded


In [ ]:
# ============================================================================
# STEP 6: COMPLETE GAME24 SOLVER CLASS (WITH ALL FEATURES)
# ============================================================================

class Game24TreeOfThoughts:
    """Complete Tree of Thoughts solver with ALL features from OpenAI version"""
    
    def __init__(self, 
                 temperature: float = 0.7,
                 n_evaluate_sample: int = 3,
                 n_select_sample: int = 15,
                 max_steps: int = 6,
                 api_delay: float = 3.5,
                 selection_method: str = 'greedy',
                 exhaustive_depth1: bool = False,
                 enable_deadend_memory: bool = True,
                 max_thread_workers: int = 5,
                 deadend_test_mode: bool = False):
        """Initialize the ToT solver with CodeAct and optional Dead-End Memory"""
        self.temperature = temperature
        self.n_evaluate_sample = n_evaluate_sample
        self.n_select_sample = n_select_sample
        self.max_steps = max_steps
        self.api_delay = api_delay
        self.selection_method = selection_method
        self.exhaustive_depth1 = exhaustive_depth1
        self.enable_ser = False
        self.enable_deadend_memory = enable_deadend_memory
        self.value_cache = {}
        self.max_thread_workers = max_thread_workers
        self.deadend_test_mode = deadend_test_mode

        # Tree structure
        self.root = None
        self.all_nodes = []
        self.solutions = []
        
        # Dead-End Memory (TIM Idea #7)
        self.dead_end_memory = DeadEndMemory(similarity_threshold=0.5, test_mode=deadend_test_mode) if enable_deadend_memory else None

        # Inconsistency Detector (TIM Idea #4)
        self.inconsistency_detector = InconsistencyDetector(inconsistency_threshold=0.3)
        
        # Statistics
        self.stats = {
            'total_nodes': 0,
            'api_calls': 0,
            'cache_hits': 0,
            'solutions_found': 0,
            'code_executions': 0,
            'code_errors': 0,
            'daily_requests': 0,
            'session_start': datetime.now(),
            'deadend_memory_enabled': enable_deadend_memory,
            'deadend_memory_skipped': 0,
            'total_input_tokens': 0,
            'total_output_tokens': 0
        }
        
        # Free tier limits
        self.DAILY_LIMIT = 14000
        self.MINUTE_LIMIT = 20
        
        print(f"✓ Solver initialized")
        print(f"  • Temperature: {temperature}")
        print(f"  • Beam width: {n_select_sample}")
        print(f"  • Max steps: {max_steps}")
        print(f"  • Dead-End Memory: {'ENABLED ✓' if enable_deadend_memory else 'DISABLED'}")
        print(f"  • Inconsistency Detection (TIM Idea #4): ENABLED ✓")
        if exhaustive_depth1:
            print(f"  • Mode: EXHAUSTIVE DEPTH-1")
        if enable_ser:
            print(f"  • Mode: SER ENABLED")
    
    def check_rate_limits(self):
        """Check if we're approaching rate limits"""
        if self.stats['daily_requests'] >= self.DAILY_LIMIT * 0.9:
            print(f"⚠ WARNING: Approaching daily limit ({self.stats['daily_requests']}/{self.DAILY_LIMIT})")
    
    def generate_all_first_moves(self, numbers: List[float]) -> List[Dict]:
        """Generate ALL possible first moves exhaustively (~24 moves)"""
        proposals = []
        operations = [
            ('+', lambda a, b: a + b, "{} + {} = {}"),
            ('-', lambda a, b: a - b, "{} - {} = {}"),
            ('*', lambda a, b: a * b, "{} × {} = {}"),
            ('/', lambda a, b: a / b if b != 0 else None, "{} ÷ {} = {}")
        ]
        
        print(f"  🔬 Exhaustively generating all first moves from {numbers}...")
        
        for i, j in combinations(range(len(numbers)), 2):
            a, b = numbers[i], numbers[j]
            remaining_indices = [k for k in range(len(numbers)) if k not in [i, j]]
            remaining = [numbers[k] for k in remaining_indices]
            
            for op_symbol, op_func, math_template in operations:
                for num1, num2, idx1, idx2 in [(a, b, i, j), (b, a, j, i)]:
                    if op_symbol in ['+', '*'] and (num1, num2) != (a, b):
                        continue
                    
                    result = op_func(num1, num2)
                    if result is None or not math.isfinite(result):
                        continue
                    
                    new_state = [result] + remaining
                    math_str = math_template.format(num1, num2, result)
                    thought = f"Exhaustive: {math_str}"
                    
                    code = f"""numbers = {numbers}
res = numbers[{idx1}] {op_symbol} numbers[{idx2}]
remaining = {remaining}
new_numbers = [res] + remaining
print(new_numbers)"""
                    _op_match = re.search(r'#\s*(.+?=.+?)$', code, re.MULTILINE)
                    _action = _op_match.group(1).strip() if _op_match else str(new_state)
                    proposal = {
                        'thought': thought,
                        'code': code,
                        'observation': str(new_state),
                        'new_state': new_state,
                        'action': _action
                    }
                    proposals.append(proposal)
        
        print(f"  ✓ Generated {len(proposals)} exhaustive first moves")
        return proposals
    
    def passes_basic_heuristics(self, move: Dict) -> bool:
        """
        Quick heuristic check to filter obviously bad moves.
        Used in hybrid SER to identify promising moves for LLM evaluation.
        """
        new_state = move['new_state']
        
        # Filter out moves that create problematic states
        if len(new_state) != 3:
            return True  # Keep depth 1 moves
        
        # Don't evaluate moves that create extreme imbalances at depth 1
        max_val = max(abs(x) for x in new_state)
        non_zero = [abs(x) for x in new_state if abs(x) > 1e-9]
        min_val = min(non_zero) if non_zero else max_val
        
        # Skip obviously problematic ratios (>1000x)
        if max_val / min_val > 1000:
            return False
        
        # Skip if creates huge numbers (>500)
        if max_val > 500:
            return False
        
        # Skip if creates tiny fractions (<0.01)
        if any(0 < abs(x) < 0.01 for x in new_state):
            return False
        
        return True
    
    def heuristic_score_move(self, move: Dict) -> float:
        """
        Heuristic scoring for moves when LLM evaluation is skipped.
        Based on distance to 24 and balance.
        """
        new_state = move['new_state']
        
        if len(new_state) != 3:
            return 1.0
        
        # Score based on how close sum/product are to 24
        sum_val = sum(new_state)
        try:
            prod_val = abs(new_state[0] * new_state[1] * new_state[2])
        except:
            prod_val = float('inf')
        
        sum_dist = abs(sum_val - 24)
        prod_dist = abs(prod_val - 24) if prod_val < 10000 else 10000
        
        # Closer to 24 gets higher score
        sum_score = 1.0 / (1.0 + sum_dist)
        prod_score = 1.0 / (1.0 + prod_dist)
        
        return max(sum_score, prod_score)
    
    def propose(self, current_numbers: List[int], original_input: List[int], history: str = "", n_proposals: int = 5, current_depth: int = 0) -> List[Dict]:
        log_to_file(f"\n=== PROPOSE for {current_numbers} depth {current_depth} ===")
        
        if len(current_numbers) == 2:
            log_to_file("→ Routing to propose_two_number")
            return self.propose_two_number(current_numbers, original_input, history)
        
        proposals = []
        seen_states = set()
        
        prompt = PROPOSE_PROMPT_CODEACT.format(
            original_input=original_input,
            history=history if history else "(Starting state)",
            input=current_numbers
        )
        
        if self.dead_end_memory is not None and current_depth >= 1:
            pattern_summary = self.dead_end_memory.get_pattern_summary(max_patterns=3)
            if pattern_summary:
                prompt += f"\n\n{pattern_summary}\n\nGiven these learned failures, generate proposals that AVOID these dead-end patterns."
                log_to_file(f"  Added dead-end patterns to prompt")
        
        self.check_rate_limits()
        time.sleep(self.api_delay)
        self.stats['api_calls'] += 1
        self.stats['daily_requests'] += 1
        
        log_to_file(f"  Calling LLM (temp={self.temperature})...")
        response = deepseek_generate(prompt, n=1, temperature=self.temperature, stats=self.stats)[0]
        log_to_file(f"  Response length: {len(response)}")
        log_to_file(f"  Response first 600 chars:\n{response[:600]}")
        
        if not response or not response.strip():
            log_to_file("  ERROR: Empty response from LLM")
            return []
        
        # ----- PARSING -----
        full_pattern = r"Thought:\s*(.+?)\s*Math:\s*(.+?)\s*Remaining:\s*(.+?)(?:Code:.*?```python\n(.*?)\n```)"
        matches = re.findall(full_pattern, response, re.DOTALL | re.IGNORECASE)
        log_to_file(f"  Full pattern matches: {len(matches)}")
        
        if not matches:
            simple_pattern = r"Thought:\s*(.+?)(?:Code:.*?```python\n(.*?)\n```)"
            matches = re.findall(simple_pattern, response, re.DOTALL | re.IGNORECASE)
            log_to_file(f"  Simple pattern matches: {len(matches)}")
            matches = [(m[0], "", "", m[1]) for m in matches]
        
        if not matches:
            log_to_file("  No code blocks found – returning []")
            return []
        
        for i, (thought, math_str, remaining_str, code) in enumerate(matches):
            if len(proposals) >= n_proposals:
                break
            code = code.strip()
            if not code:
                log_to_file(f"  Match {i}: no code, skipped")
                continue
            
            log_to_file(f"  Match {i}: code snippet:\n{code[:200]}")
            try:
                self.stats['code_executions'] += 1
                sandbox.globals['numbers'] = list(current_numbers)
                observation, new_state = sandbox.run(code)
                
                if new_state and not observation.startswith("Error:"):
                    state_tuple = tuple(sorted(new_state))
                    if state_tuple not in seen_states:
                        proposals.append({
                            'thought': thought,
                            'code': code,
                            'observation': observation,
                            'new_state': new_state,
                            'action': observation
                        })
                        seen_states.add(state_tuple)
                        log_to_file(f"    ✓ Valid proposal: {new_state}")
                    else:
                        log_to_file(f"    ✗ Duplicate state: {new_state}")
                else:
                    log_to_file(f"    ✗ Execution error: {observation[:100]}")
            except Exception as e:
                log_to_file(f"    ✗ Exception: {e}")
        
        log_to_file(f"  Returning {len(proposals)} proposals")
        return proposals
        
    def propose_two_number(self, current_numbers: List, original_input: str, history: str = "") -> List[dict]:
        """LLM-only proposal for 2-number terminal states. Returns 1 proposal or empty list."""
        proposals = []
        
        if len(current_numbers) != 2:
            return proposals
        
        a, b = current_numbers[0], current_numbers[1]
        
        # Pre-calculate results for the prompt (optional, but helps LLM)
        sum_res = a + b
        diff_ab = a - b
        diff_ba = b - a
        prod_res = a * b
        div_ab = a / b if b != 0 else None
        div_ba = b / a if a != 0 else None
        
        prompt_content = TWO_NUMBER_PROPOSE_PROMPT.format(
            original_input=original_input,
            history=history if history else "No prior steps",
            input=f"Current state: {current_numbers}",
            a=a, b=b,
            sum_result=sum_res,
            diff_ab=diff_ab,
            diff_ba=diff_ba,
            prod_result=prod_res,
            div_ab=div_ab if div_ab is not None else "undefined",
            div_ba=div_ba if div_ba is not None else "undefined"
        )
        
        # Call LLM (single attempt, no fallback)
        try:
            response = deepseek_generate(prompt_content, n=1, temperature=self.temperature, stats=self.stats)[0]
        except Exception as e:
            print(f"      [DEBUG] LLM call failed for {current_numbers}: {e}")
            return proposals  # empty list – no child
        
        # Parse the operation from LLM response
        operation_patterns = [
            r"Selected operation:\s*(.+?)(?:\n|$)",
            r"Operation:\s*(.+?)(?:\n|$)",
            r"Best operation:\s*(.+?)(?:\n|$)",
            r"Recommended:\s*(.+?)(?:\n|$)"
        ]
        operation_str = None
        for pattern in operation_patterns:
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                operation_str = match.group(1).strip()
                break
        
        if not operation_str:
            print(f"      [DEBUG] No valid operation found in LLM response for {current_numbers}")
            return proposals  # no fallback – fail silently
        
        # Build and execute the code for that operation
        code = f"numbers = {list(current_numbers)}\n"
        code += f"a, b = {a}, {b}\n"
        code += f"result = {operation_str}\n"
        code += "new_numbers = [x for x in numbers if x not in [a, b]] + [result]\n"
        code += "new_numbers.sort()\n"
        
        try:
            sandbox.globals['numbers'] = list(current_numbers)
            observation, new_state = sandbox.run(code)
            if new_state and not observation.startswith("Error:"):
                proposals.append({
                    'thought': f"For 2-number state {current_numbers}, LLM chose: {operation_str}",
                    'code': code,
                    'observation': observation,
                    'new_state': new_state,
                    'action': observation
                })
                print(f"      [DEBUG] ✓ LLM-only proposal created: {new_state} via {operation_str}")
            else:
                print(f"      [DEBUG] ✗ Execution failed for {operation_str}: {observation}")
        except Exception as e:
            print(f"      [DEBUG] ✗ Exception: {e}")
        
        return proposals
    
    def evaluate_state(self, numbers: List[int], is_final: bool = False) -> Tuple[float, dict]:
        """HYBRID EVALUATION: Heuristics + LLM"""
        eval_record = {
            "state": str(numbers),
            "is_final": is_final,
            "heuristic_checks": {},
            "llm_judgments": [],
            "reasoning": [],
            "score_breakdown": {},
            "final_value": 0.0
        }
        
        # 1. Check if final answer
        if len(numbers) == 1:
            eval_record["heuristic_checks"]["is_single_number"] = True
            if abs(numbers[0] - 24) < 0.001:
                eval_record["heuristic_checks"]["is_solution"] = True
                eval_record["reasoning"].append("✅ SOLUTION: Final state equals 24 - PUZZLE SOLVED!")
                eval_record["final_value"] = 1.0
                return 1.0, eval_record
            else:
                eval_record["heuristic_checks"]["is_solution"] = False
                eval_record["reasoning"].append(f"❌ WRONG ANSWER: Final state is {numbers[0]}, not 24")
                eval_record["final_value"] = 0.001
                return 0.001, eval_record
        
        # 2. Penalize premature 24
        if 24 in numbers or 24.0 in numbers:
            eval_record["heuristic_checks"]["has_premature_24"] = True
            eval_record["reasoning"].append("⚠️ DEAD-END: Contains 24 but not in final state - dead-end trap!")
            eval_record["final_value"] = 0.01
            return 0.01, eval_record
        else:
            eval_record["heuristic_checks"]["has_premature_24"] = False
        
        # 3. Penalize huge numbers
        max_abs = max(abs(n) for n in numbers)
        eval_record["heuristic_checks"]["max_abs_value"] = max_abs
        if max_abs > 1000:
            eval_record["heuristic_checks"]["has_huge_numbers"] = True
            eval_record["reasoning"].append(f"⚠️ HUGE NUMBERS: Max value {max_abs} exceeds 1000 - unlikely to reach 24")
            eval_record["final_value"] = 0.1
            return 0.1, eval_record
        
        
        
        # 5. LLM Evaluation for 3+ numbers
        numbers_str = str(numbers)
        prompt = VALUE_PROMPT_CODEACT.format(input=numbers_str)
        
        if prompt in self.value_cache:
            self.stats['cache_hits'] += 1
            cached_value = self.value_cache[prompt]
            eval_record["from_cache"] = True
            eval_record["final_value"] = cached_value
            return cached_value, eval_record
        
        self.check_rate_limits()
        time.sleep(self.api_delay)
        self.stats['api_calls'] += self.n_evaluate_sample
        self.stats['daily_requests'] += self.n_evaluate_sample
        
        value_outputs = []
        for i in range(self.n_evaluate_sample):
            try:
                response = deepseek_generate(prompt, n=1, temperature=0.0, stats=self.stats)[0]
                value_outputs.append(response.strip().lower())
            except Exception as e:
                value_outputs.append("") 
        
        # Score mapping: sure > likely > impossible (proper differentiation)
        # sure=1.0 (highest confidence), likely=0.6 (medium), impossible=0.001 (lowest)
        value_map = {'impossible': 0.001, 'likely': 0.6, 'sure': 1.0}
        value_names = []
        
        for raw in value_outputs:
            if not raw:
                value_names.append("likely")
                continue
            # Try to find "ANSWER: sure/likely/impossible"
            match = re.search(r"ANSWER:\s*(sure|likely|impossible)", raw, re.IGNORECASE)
            if match:
                judgment = match.group(1).lower()
            else:
                # Fallback: last word of response
                words = raw.strip().split()
                judgment = words[-1].lower() if words else "likely"
                if judgment not in ["sure", "likely", "impossible"]:
                    judgment = "likely"
            value_names.append(judgment)
        
        # Store LLM judgments and reasoning
        eval_record["llm_judgments"] = value_outputs  # Raw responses
        eval_record["score_breakdown"] = {
            "raw_responses": value_outputs,
            "mapped_values": [value_map.get(name, 0.6) for name in value_names],
            "vote_counts": {name: value_names.count(name) for name in value_map.keys()}
        }
        
        # Add detailed reasoning
        eval_record["reasoning"].append(f"LLM evaluations: {value_names}")
        eval_record["reasoning"].append(f"Judgment votes: sure={value_names.count('sure')}, likely={value_names.count('likely')}, impossible={value_names.count('impossible')}")
        from collections import Counter
        final_judgment = Counter(value_names).most_common(1)[0][0]
        eval_record['judgment'] = final_judgment
        
        # Aggregate votes: average the mapped values (not sum which was problematic)
        raw_value = sum(value_map.get(name, 0.6) for name in value_names) / len(value_names) if value_names else 0.6
        boosted_value = raw_value * 1.2 if any(20 <= abs(n) <= 40 for n in numbers) and any(1 <= abs(n) <= 10 for n in numbers) else raw_value
        
        eval_record["reasoning"].append(f"Raw score: {raw_value:.3f}, Boosted score: {boosted_value:.3f}")
        eval_record["final_value"] = boosted_value
        self.value_cache[prompt] = boosted_value
        
        # NEW: Track evaluation for inconsistency detection (TIM Idea #4)
        if self.inconsistency_detector is not None:
            inconsistency_info = self.inconsistency_detector.record_evaluation(
                state=numbers,
                score=boosted_value,
                depth=getattr(self, 'current_depth', 0),
                reasoning=eval_record.get('reasoning', '')[-1] if eval_record.get('reasoning') else ''
            )
            if inconsistency_info and inconsistency_info.get('is_inconsistent'):
                eval_record['inconsistency_warning'] = {
                    'variance': inconsistency_info['variance'],
                    'scores': inconsistency_info['scores']
                }
        extracted_op = None
        extracted_result = None
        if len(numbers) == 2 and value_outputs:
            raw = value_outputs[0]  # first sample
            # Pattern: e.g., "28 + (-4) = 24" or "28 - 4 = 24"
            # Captures: a, operator, b, result
            # Scan ALL matches, prefer the one whose result is 24
            pattern_with_result = r'([+-]?\d+(?:\.\d+)?)\s*([+\-*/])\s*([+-]?\d+(?:\.\d+)?)\s*=\s*(\d+(?:\.\d+)?)'
            for match in re.finditer(pattern_with_result, raw):
                a_str, op, b_str, res_str = match.groups()
                if abs(float(res_str) - 24.0) < 0.001:
                    extracted_op = f"{a_str} {op} {b_str}"
                    extracted_result = 24.0
                    break
            # Fallback: take first match if none equalled 24
            if extracted_op is None:
                match = re.search(pattern_with_result, raw)
                if match:
                    a_str, op, b_str, res_str = match.groups()
                    extracted_op = f"{a_str} {op} {b_str}"
                    extracted_result = float(res_str)
            
            # Post-LLM verifier: if LLM claims this 2-number state reaches 24, check the arithmetic
        if len(numbers) == 2 and extracted_result is not None and abs(extracted_result - 24) < 0.001:
            try:
                m = re.match(r'([+-]?\d+(?:\.\d+)?)\s*([+\-*/])\s*([+-]?\d+(?:\.\d+)?)', extracted_op)
                ops_map = {'+': lambda a,b: a+b, '-': lambda a,b: a-b,
                        '*': lambda a,b: a*b, '/': lambda a,b: a/b}
                actual = ops_map[m.group(2)](float(m.group(1)), float(m.group(3)))

                # also verify both operands are actually in the state
                operands = sorted([abs(float(m.group(1))), abs(float(m.group(3)))])
                state_vals = sorted([abs(x) for x in numbers])
                operands_valid = all(abs(operands[i] - state_vals[i]) < 0.001 for i in range(2))

                if abs(actual - 24) > 0.001 or not operands_valid:
                    eval_record['extracted_result'] = None
                    eval_record['extracted_operation'] = None
                    eval_record['reasoning'].append(
                        f"❌ VERIFIER: LLM claimed '{extracted_op}=24' but actual={actual:.4f}, "
                        f"operands_valid={operands_valid} — closing node"
                    )
                    return 0.001, eval_record

                eval_record['reasoning'].append(f"✅ VERIFIER: '{extracted_op}=24' confirmed")

            except Exception as e:
                eval_record['extracted_result'] = None
                eval_record['extracted_operation'] = None
                eval_record['reasoning'].append(f"❌ VERIFIER: parse error ({e}) — closing node")
                return 0.001, eval_record


        eval_record['extracted_operation'] = extracted_op
        eval_record['extracted_result'] = extracted_result
        
        return boosted_value, eval_record
    
    def solve(self, numbers: List[int], verbose: bool = True) -> Tuple[List[str], 'TreeNode']:
        print(f"\n[SOLVE START] Input: {numbers}, verbose: {verbose}")
        def evaluate_child(child):
            value, eval_record = self.evaluate_state(child.state, is_final=(len(child.state) == 1))
            return child, value, eval_record
        def is_duplicate_solution(parent, op):
            for sol in self.solutions:
                if sol.parent == parent and sol.action == op:
                    return True
            return False
        
        TreeNode.node_counter = 0
        self.all_nodes = []
        self.solutions = []
        original_input = numbers.copy()
        self.original_input = original_input
        self.dead_end_memory = DeadEndMemory.load("dead_end_db.json")
        self.dead_end_memory.current_problem_id = str(numbers)   # e.g. "[3,3,6,13]"
        
        # Create root
        self.root = TreeNode(state=numbers, depth=0)
        self.all_nodes.append(self.root)
        
        # Track global seen states
        global_seen_states = set()
        global_seen_states.add(tuple(sorted(numbers)))
        
        queue = [(self.root, "")]
        
        for depth in range(self.max_steps):
            if verbose:
                print(f"\n{'='*70}")
                print(f"STEP {depth + 1}/{self.max_steps}")
                print(f"Current candidates: {len(queue)}")
            
            next_queue = []
            
            # === HYBRID SER MODE at DEPTH 0 ===
            if self.enable_ser and depth == 0:
                pass  #disable
            
            # === NORMAL LLM PROPOSALS MODE (depth >= 1) ===
            else:
                print(f"  [DEBUG] Entering LLM proposal mode for depth {depth}")
                print(f"  [DEBUG] Queue has {len(queue)} nodes to process")
                
                # Process EVERY node in the current queue
                for node_idx, (node, history) in enumerate(queue):
                    print(f"\n  [DEBUG] Processing queue item {node_idx}: node.state={node.state}, len={len(node.state)}")
                    
                    if len(node.state) == 1:
                        print(f"  [DEBUG]   Skipping: state is single number")
                        continue
                    
                    if verbose:
                        print(f"\n  Node {node.id}: Generating proposals for {node.state}")
                    
                    proposals = self.propose(
                        node.state,
                        original_input=original_input,
                        history=history,
                        n_proposals=8,
                        current_depth=depth
                    )
                    
                    if verbose:
                        print(f"    → Generated {len(proposals)} unique proposals")
                    
                                        # Add ALL children from this node to next_queue
                    debug_log(f"\n[DEBUG] Node {node.id}: processing {len(proposals)} proposals")
                    for prop in proposals:
                        new_state = prop['new_state']
                        state_tuple = tuple(sorted(new_state))
                        debug_log(f"  Proposal: {new_state} -> tuple {state_tuple}")
                        
                        if state_tuple in global_seen_states:
                            debug_log(f"    -> DUPLICATE, skipping (already seen)")
                            continue
                        
                        if self.dead_end_memory is not None:
                            is_dead_end, matched_pattern = self.dead_end_memory.is_potential_dead_end(new_state)
                            if is_dead_end:
                                debug_log(f"    -> DEAD-END pattern, skipping (reason: {matched_pattern.get('reason', 'unknown')})")
                                if verbose:
                                    print(f"      ⚠️  Skipping {new_state} - matches dead-end pattern (reason: {matched_pattern.get('reason', 'unknown')})")
                                self.stats['deadend_memory_skipped'] += 1
                                continue
                        
                        global_seen_states.add(state_tuple)
                        debug_log(f"    -> Added to global_seen_states")
                        
                        step_desc = f"{prop['thought']}\nResult: {prop['observation']}"
                        new_history = history + "\n" + step_desc if history else step_desc
                        
                        child = TreeNode(
                            state=new_state,
                            parent=node,
                            action=prop['action'],
                            depth=depth + 1
                        )
                        child.thought = prop['thought']
                        child.code = prop['code']
                        child.observation = prop['observation']
                        child.path_history = new_history
                        
                        node.children[child.id] = child
                        next_queue.append((child, new_history))
                        self.all_nodes.append(child)
                        debug_log(f"    -> Child {child.id} created and added to next_queue")
                
                print(f"  [DEBUG] next_queue size: {len(next_queue)}")
                
                if not next_queue:
                    if verbose:
                        print("\n  ⚠ No new proposals. Stopping.")
                    break
                
                # Evaluate all new nodes (ALL children from ALL parents)
                if verbose:
                    print(f"\n  Evaluating {len(next_queue)} new states...")

                with ThreadPoolExecutor(max_workers=self.max_thread_workers) as executor:  # adjust workers; 5 is safe for free tier
                    futures = {executor.submit(evaluate_child, child): child for child, _ in next_queue}
                    for future in as_completed(futures):
                        child, value, eval_record = future.result()
                        child.value = value
                        child.evaluation = eval_record
                    
                        # 2‑number handling: if a terminal 2‑number state can reach 24, create the final node directly
                        if len(child.state) == 2 and eval_record.get('extracted_result') is not None and abs(eval_record['extracted_result'] - 24.0) < 0.001:
                            op_str = eval_record['extracted_operation']
                            if op_str and verbose:
                                print(f"      [2-NUMBER] {child.state} → {op_str} = 24. Creating final node.")
                            
                            # Build code for the operation
                            a, b = child.state[0], child.state[1]
                            code = f"numbers = {list(child.state)}\n"
                            code += f"a, b = {a}, {b}\n"
                            code += f"result = {op_str}\n"
                            code += "new_numbers = [result]"
                            
                            step_desc = f"Operation: {op_str} = 24"
                            new_history = child.path_history + "\n" + step_desc if child.path_history else step_desc
                            
                            solution_node = TreeNode(
                                state=[24.0],
                                parent=child,
                                action=op_str,
                                depth=child.depth + 1
                            )
                            solution_node.is_solution = True
                            solution_node.value = 1.0
                            solution_node.thought = f"From {child.state}, {op_str} = 24"
                            solution_node.code = code
                            solution_node.observation = "[24]"
                            solution_node.path_history = new_history
                            
                            child.children[solution_node.id] = solution_node
# Dedup: only register if no solution already exists with this parent
                            if not any(s.parent and s.parent.id == child.id for s in self.solutions):
                                self.solutions.append(solution_node)
                            self.all_nodes.append(solution_node)
                           
                            
                            if verbose:
                                print(f"      ✅ SOLUTION FOUND: {op_str} = 24")
                            # Do NOT add this child to next_queue (it's already resolved)
                            continue
                
                # Debug: show which nodes are being considered (first 10)
                print(f"  [DEBUG] Before global sort (first 10 of {len(next_queue)}):")
                for i in range(min(10, len(next_queue))):
                    node, _ = next_queue[i]
                    print(f"    {i+1}. Node {node.id} (parent {node.parent.id}): value={node.value:.3f}, state={node.state}")
                
                # Sort and select top‑k from ALL children together
                next_queue.sort(key=lambda x: x[0].value, reverse=True)
                selected = next_queue[:self.n_select_sample]
                
                # Record patterns from pruned nodes for Dead‑End Memory
                if self.dead_end_memory is not None:
                    pruned_nodes = next_queue[self.n_select_sample:]
                    for node, _ in pruned_nodes:
                        if node.value < 1.0:
                            reason = f"pruned_value_{node.value:.3f}"
                            self.dead_end_memory.record_dead_end(node, reason)
                
                queue = selected
                
                if verbose:
                    print(f"\n  Selected top {len(queue)} candidates for next depth:")
                    for i, (node, _) in enumerate(queue[:3]):
                        print(f"    {i+1}. Value={node.value:.2f} | State={node.state}")
        
        # Collect final solutions
        for node in self.all_nodes:
            if len(node.state) == 1 and abs(node.state[0] - 24) < 0.001:
                node.is_solution = True
                if node not in self.solutions and not any(
                    s.parent and s.parent.id == node.parent.id for s in self.solutions
                ):
                    self.solutions.append(node)
        
        self.stats['total_nodes'] = len(self.all_nodes)
        self.stats['solutions_found'] = len(self.solutions)
        
        if self.dead_end_memory is not None:
            self.stats['deadend_memory'] = self.dead_end_memory.get_stats()
            self.stats['deadend_memory_skipped'] = self.dead_end_memory.total_skipped
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✓ Found {len(self.solutions)} solution(s)")
            print(f"  Total nodes: {self.stats['total_nodes']}")
            print(f"  API calls: {self.stats['api_calls']}")
            print(f"  Cache hits: {self.stats['cache_hits']}")
            if self.dead_end_memory is not None:
                mem_stats = self.dead_end_memory.get_stats()
                print(f"  Dead-End Memory:")
                print(f"    - Patterns stored: {mem_stats['patterns_stored']}")
                print(f"    - States checked: {mem_stats['total_checked']}")
                print(f"    - States skipped: {mem_stats['total_skipped']} ({mem_stats['skip_rate']})")
                api_saved = mem_stats['total_skipped'] * self.n_evaluate_sample
                print(f"    - API calls saved: {api_saved}")
        
        print(f"[SOLVE END]\n")
        if self.dead_end_memory is not None:
            self.dead_end_memory.save("dead_end_db.json")
        return [self.reconstruct_solution_path(node) for node in self.solutions], self.root
    
    def reconstruct_solution_path(self, node: 'TreeNode') -> str:
        """Reconstruct the solution path from root to node"""
        path = []
        current = node
        
        while current.parent is not None:
            step_info = f"Thought: {current.thought}\nCode:\n{current.code}\nResult: {current.observation}"
            path.append(step_info)
            current = current.parent
        
        path.reverse()
        return "\n\n".join(path)
    
    def export_tree_to_json(self, filename: str = None) -> str:
        """Export the entire search tree to JSON"""
        import os
        
        # Create raw_tree folder if it doesn't exist
        raw_tree_folder = "raw_tree"
        if not os.path.exists(raw_tree_folder):
            os.makedirs(raw_tree_folder)
            print(f"✓ Created folder: {raw_tree_folder}")
        
        if filename is None:
            # Include the problem in the filename
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            if hasattr(self, 'original_input') and self.original_input:
                problem_str = "_".join(str(n) for n in self.original_input)
                filename = f"{raw_tree_folder}/game24_tree_{problem_str}_{timestamp}.json"
            else:
                filename = f"{raw_tree_folder}/game24_tree_{timestamp}.json"
        
        stats_serializable = self.stats.copy()
        if 'session_start' in stats_serializable:
            try:
                stats_serializable['session_start'] = stats_serializable['session_start'].isoformat()
            except AttributeError:
                pass
        
        # NEW: Add Dead-End Memory stats to export
        mode = 'CodeAct with Dead-End Memory' if self.enable_deadend_memory else 'CodeAct'
        
        tree_data = {
            'metadata': {
                'timestamp': datetime.now().isoformat(),
                'mode': mode,
                'parameters': {
                    'temperature': self.temperature,
                    'n_evaluate_sample': self.n_evaluate_sample,
                    'n_select_sample': self.n_select_sample,
                    'max_steps': self.max_steps,
                    'api_delay': self.api_delay,
                    'enable_deadend_memory': self.enable_deadend_memory
                },
                'statistics': stats_serializable,
                'deadend_memory_summary': {
                    'enabled': self.enable_deadend_memory,
                    'patterns_stored': len(self.dead_end_memory.patterns) if self.dead_end_memory else 0,
                    'total_checked': self.dead_end_memory.total_checked if self.dead_end_memory else 0,
                    'total_skipped': self.dead_end_memory.total_skipped if self.dead_end_memory else 0,
                    'api_calls_saved': (self.dead_end_memory.total_skipped * self.n_evaluate_sample) if self.dead_end_memory else 0
                }
            },
            'nodes': [node.to_dict() for node in self.all_nodes],
            'solutions': [node.id for node in self.solutions],
            # NEW: Add Inconsistency Detection Report (TiM Idea #4)
            'inconsistency_report': {
                'enabled': self.inconsistency_detector is not None,
                'statistics': self.inconsistency_detector.get_stats() if self.inconsistency_detector else {},
                'inconsistent_states': self.inconsistency_detector.get_inconsistent_states() if self.inconsistency_detector else [],
                'suspicious_states': self.inconsistency_detector.get_suspicious_states() if self.inconsistency_detector else []
            }
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(tree_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Tree exported to: {filename}")
        if self.enable_deadend_memory and self.dead_end_memory:
            print(f"  • Dead-End Memory: {self.dead_end_memory.total_skipped} states skipped")
            print(f"  • API calls saved: {self.dead_end_memory.total_skipped * self.n_evaluate_sample}")
        return filename

print("✓ Game24TreeOfThoughts COMPLETE solver class loaded")
print("✓ Includes: SER, exhaustive_depth1, DFP, global tracking, hybrid evaluation, Dead-End Memory")

In [7]:
# ============================================================================
# STEP 7: VISUALIZATION & ANALYSIS FUNCTIONS
# ============================================================================

def visualize_tree_codeact(root_node, max_depth=3):
    """Visualize the CodeAct search tree structure"""
    def print_node(node, depth=0, prefix="", is_last=True):
        if depth > max_depth:
            return
        
        branch = "└── " if is_last else "├── "
        extension = "    " if is_last else "│   "
        
        if node.depth == 0:
            line = f"🌳 ROOT: {node.state or 'Start'}"
        else:
            action = node.action[:50] + "..." if len(node.action) > 50 else node.action
            value_str = f"(value: {node.value:.2f})" if node.value > 0 else ""
            solution_marker = " ✅ SOLUTION" if node.is_solution else ""
            pruned_marker = " ❌ PRUNED" if node.is_pruned else ""
            
            line = f"{branch}[D{node.depth}] {action} {value_str}{solution_marker}{pruned_marker}"
        
        print(prefix + line)
        
        children = list(node.children.values())
        for i, child in enumerate(children):
            is_last_child = (i == len(children) - 1)
            new_prefix = prefix + extension
            print_node(child, depth + 1, new_prefix, is_last_child)
    
    print("\n" + "="*70)
    print("🎄 SEARCH TREE VISUALIZATION")
    print("="*70)
    print_node(root_node)
    print("="*70 + "\n")

def analyze_tree(root_node):
    """Analyze and display statistics about the search tree"""
    stats = {
        'total_nodes': 0,
        'by_depth': {},
        'solutions': 0,
        'pruned': 0,
        'max_depth': 0,
        'avg_branching': 0,
        'total_children': 0
    }
    
    def traverse(node):
        stats['total_nodes'] += 1
        depth = node.depth
        
        if depth not in stats['by_depth']:
            stats['by_depth'][depth] = 0
        stats['by_depth'][depth] += 1
        
        if depth > stats['max_depth']:
            stats['max_depth'] = depth
        
        if node.is_solution:
            stats['solutions'] += 1
        if node.is_pruned:
            stats['pruned'] += 1
        
        num_children = len(node.children)
        if num_children > 0:
            stats['total_children'] += num_children
        
        for child in node.children.values():
            traverse(child)
    
    traverse(root_node)
    
    internal_nodes = stats['total_nodes'] - stats['by_depth'].get(stats['max_depth'], 0)
    stats['avg_branching'] = stats['total_children'] / internal_nodes if internal_nodes > 0 else 0
    
    print("\n" + "="*70)
    print("📊 TREE ANALYSIS")
    print("="*70)
    print(f"Total Nodes:        {stats['total_nodes']}")
    print(f"Max Depth:          {stats['max_depth']}")
    print(f"Solutions Found:    {stats['solutions']}")
    print(f"Pruned Nodes:       {stats['pruned']}")
    print(f"Avg Branching:      {stats['avg_branching']:.2f}")
    print()
    print("Nodes by Depth:")
    for depth in sorted(stats['by_depth'].keys()):
        count = stats['by_depth'][depth]
        bar = "█" * min(count, 50)
        print(f"  Depth {depth}: {count:3d} {bar}")
    print("="*70 + "\n")

print("✓ Visualization and analysis functions loaded")

✓ Visualization and analysis functions loaded


---

## 🎯 Ready to Test!

You've completed the setup. Now you can:

1. **Create a solver instance** (see cells below)
2. **Test with example puzzles**
3. **View results and tree analysis**

---

In [15]:
# ============================================================================
# EXAMPLE: Solve Game of 24 puzzle [1, 2, 4, 7]
# ============================================================================
# Solution: (7-2+1)*4 = 6*4 = 24

input_numbers = [6,9, 9, 10]

print("="*70)
print("🎮 GAME OF 24 SOLVER - TEST PUZZLE")
print("="*70)
print(f"\n📊 Input Numbers: {input_numbers}")
print(f"🎯 Goal: Use +, -, *, / to make 24")
print(f"💡 Solution: (7-2+1)*4 = 24")
print(f"\n⚙️  Solver Configuration:")
print(f"   • Temperature: 0.7 (balanced exploration)")
print(f"   • Evaluation samples: 3 (per state)")
print(f"   • Beam width: 10 (keep top 10 candidates per level)")
print(f"   • Max search depth: 6 (up to 6 operations)")
print(f"   • Rate limit: 3.5s between API calls")
print(f"   • Exhaustive first moves: OFF (use LLM proposals)")
print(f"   • SER (Selective Exhaustive Rescue): OFF")
print(f"   • DFP (Delayed Fraction Preservation): ON")
print()

# Create solver instance
solver = Game24TreeOfThoughts(
    temperature=0.7,
    n_evaluate_sample=1,
    n_select_sample=5,
    max_steps=6,
    api_delay=API_DELAY,
    selection_method='greedy',
    enable_deadend_memory=True,
    max_thread_workers=10,
    deadend_test_mode=False
)

print(f"{'='*70}")
print(f"🚀 STARTING SEARCH...")
print(f"{'='*70}\n")

# Run the solver
solutions, root = solver.solve(input_numbers, verbose=True)

# Display results
print(f"\n{'='*70}")
print(f"RESULTS")
print(f"{'='*70}\n")

if solutions:
    print(f"✅ SUCCESS! Found {len(solutions)} solution(s)\n")
    for i, sol in enumerate(solutions, 1):
        print(f"Solution {i}:")
        print("-" * 70)
        print(sol)
        print("-" * 70)
        print()
else:
    print(f"❌ No solution found in {solver.stats['total_nodes']} nodes explored")
    print(f"\n💡 Try enabling SER or exhaustive_depth1 for harder puzzles:")
    print(f"   • exhaustive_depth1=True: Try ALL ~24 possible first moves")
    print(f"   • enable_ser=True: Rescue with exhaustive search if LLM proposals are weak")
    print()

# Display statistics
print(f"{'='*70}")
print(f"📊 STATISTICS")
print(f"{'='*70}\n")
print(f"Total nodes explored:   {solver.stats['total_nodes']}")
print(f"API calls made:         {solver.stats['api_calls']}")
print(f"Cache hits:             {solver.stats['cache_hits']}")
print(f"Code executions:        {solver.stats['code_executions']}")
print(f"Code errors:            {solver.stats['code_errors']}")
print(f"Daily requests used:    {solver.stats['daily_requests']}/14000")
print()

# Export tree to JSON
json_file = solver.export_tree_to_json()
print(f"\n📁 Complete search tree exported to: {json_file}")
print(f"\n💾 You can analyze this JSON to understand the search process:")
print(f"   • All nodes visited")
print(f"   • Parent-child relationships")
print(f"   • State evaluations and reasoning")
print(f"   • Solution paths")
print()

print(f"{'='*70}")
if solutions:
    print(f"✨ PUZZLE SOLVED! ✨")
else:
    print(f"🔧 Consider tweaking parameters for harder puzzles")
print(f"{'='*70}")